# Tutorial: Continuous S-LDSC Annotations and MAGMA Analysis

This tutorial demonstrates the new functions added to `cellink` for:

1. **Continuous S-LDSC annotations** — assign per-gene scores (e.g. from SEISMIC or cell-type specificity) to SNPs as continuous values rather than binary 0/1 flags.
2. **MAGMA gene-set analysis (GSA)** — test whether top-scoring genes per cell type are enriched for GWAS signal.
3. **MAGMA gene property analysis (GPA)** — test the linear relationship between continuous per-gene scores and GWAS gene-level z-scores.

All new functions are available at `cellink.tl.external`. MAGMA must be installed separately; download it from [https://ctg.cncr.nl/software/magma](https://ctg.cncr.nl/software/magma).

### Relationship to the previous LDSC tutorial

The companion notebook `cell_level_ldsc_analysis_updates.ipynb` covers binary S-LDSC annotations and cell-type-specific heritability (`--h2-cts`). This notebook extends that pipeline in two directions:

* **Continuous annotations** replace the binary top-10% threshold with raw score values, giving a more quantitative enrichment test.
* **MAGMA** provides an independent analysis path that does not require LD score files and can be run from scratch on any GWAS.

### Required inputs

| Input | Description |
|---|---|
| `specificity_df` | genes × cell-types DataFrame of per-gene scores (e.g. from `preprocess_for_sldsc`) |
| Gene coordinate file | Tab-separated file mapping genes to chr/start/end positions |
| PLINK bimfile (for LDSC) | Reference panel `.bim` file from e.g. 1000 Genomes |
| GWAS SNP location file (for MAGMA Step I) | SNP/CHR/BP columns |
| MAGMA gene location file | `NCBI38.gene.loc` ships with MAGMA |
| PLINK bfile (for MAGMA Step II) | LD reference panel, e.g. 1000 Genomes EUR |
| GWAS p-value file (for MAGMA Step II) | SNP + P columns |
| `.genes.raw` file (for MAGMA Step III) | Output of Step II; can be pre-computed |


## Environment Setup

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import os

from cellink._core import DAnn, GAnn
from cellink.resources import get_onek1k
from cellink.tl.external import (
    # LDSC preprocessing (same as companion notebook)
    preprocess_for_sldsc,
    generate_gene_coord_file,
    generate_sldsc_genesets,
    configure_ldsc_runner,
    # New: continuous S-LDSC annotations
    make_continuous_annot_from_bimfile,
    make_continuous_annot_from_donor_data,
    compute_ld_scores_with_annotations_from_bimfile,
    compute_ld_scores_with_annotations_from_donor_data,
    estimate_celltype_specific_heritability,
    # LDSC → MAGMA conversion
    genesets_dir_to_entrez_gmt,
    scores_to_gmt,
    scores_to_covar,
    # MAGMA pipeline from scratch
    run_magma_annotate,
    run_magma_gene_analysis,
    run_magma_gsa,
    run_magma_gpa,
)

# Paths — adjust for your environment
MAGMA_BIN = "/project/genomics/ayshan/ldsc_analysis/psychmagma/magma"
GENE_COORD_FILE = "gene_coords.txt"  # produced below by generate_gene_coord_file
MAGMA_GENE_LOC  = "/project/genomics/ayshan/ldsc_analysis/data_2/annotation_sources/NCBI38.gene.loc"
G1000_BFILE     = "/project/genomics/ayshan/ldsc_analysis/data_2/1000G_EUR_Phase3/g1000_eur"

# Cell type to use as the example throughout this tutorial
CELL_TYPE = "CD8 Naive"
celltype_key = "predicted.celltype.l2"
original_donor_col = "donor_id"

## Load and Prepare Data

We reuse the same OneK1K dataset and preprocessing steps from the companion notebook. If you already ran that notebook, load `specificity` directly from disk and skip ahead.

In [ ]:
dd = get_onek1k(
    config_path="cellink/src/cellink/resources/config/onek1k.yaml",
    data_home="/project/genomics/ayshan/1k1k_dataset",
    verify_checksum=False,
)
print(f"Dataset shape: {dd.shape}")
dd.C.obs[DAnn.donor] = dd.C.obs[original_donor_col]

In [ ]:
# Subset to ~0.1% of SNPs per chromosome for speed (tutorial only)
np.random.seed(42)
all_selected_idx = []
for chrom in range(1, 23):
    chrom_idx = np.where(dd.G.var.chrom == str(chrom))[0]
    n_snps = max(1, int(len(chrom_idx) * 0.001))
    selected_idx = np.random.choice(chrom_idx, n_snps, replace=False)
    all_selected_idx.extend(selected_idx)
dd = dd[:, np.sort(all_selected_idx), :, :].copy()

In [ ]:
dd.C.var["gene"] = dd.C.var_names
adata = dd.C.copy()

adata_filtered, mean_expr, specificity = preprocess_for_sldsc(
    adata,
    celltype_col=celltype_key,
    gene_identifier_mode="ensembl",
    gene_col="gene",
    genome_build="GRCh38",
    inplace=False,
)
print(f"Specificity shape: {specificity.shape}")
specificity.head()

In [ ]:
# Gene coordinate file (headed format: GENE / CHR / START / END)
generate_gene_coord_file(GENE_COORD_FILE, gene_identifier_mode="ensembl", genome_build="GRCh38", overwrite=True)

---
## Part 1 — Continuous S-LDSC Annotations

### Why continuous annotations?

Binary annotations assign 1 to the top 10% of genes and 0 to the rest. Continuous annotations instead assign each SNP the actual score of the highest-scoring gene whose window overlaps it (or 0 if no gene overlaps). This preserves the full quantitative signal and is more statistically powerful when gene scores vary smoothly.

Both annotation types write the same five-column file format (`CHR BP SNP CM ANNOT`), so all downstream calls (`compute_ld_scores_with_annotations_from_bimfile`, `estimate_celltype_specific_heritability`) are **identical** for both types.

### From a DonorData object

In [ ]:
runner = configure_ldsc_runner(config_path="cellink/src/cellink/tl/external/config/ldsc_singularity.yaml")

In [ ]:
ct_safe = CELL_TYPE.replace(" ", "_")

# scores: a pd.Series indexed by ENSG gene IDs
scores = specificity[CELL_TYPE]

for chrom in range(1, 23):
    dd_chrom = dd.sel(
        G_var=dd.G.var.chrom == str(chrom),
        C_var=dd.C.var.chrom == str(chrom),
    ).copy()

    result = make_continuous_annot_from_donor_data(
        dd=dd_chrom,
        scores=scores,
        annot_file=f"continuous_annots/{ct_safe}.{chrom}.annot.gz",
        gene_coord_file=GENE_COORD_FILE,
        windowsize=100_000,   # ±100 kb around each gene body
        score_agg="max",      # assign each SNP the highest overlapping gene score
    )
    print(f"chr{chrom}: {result['n_nonzero_snps']:,} non-zero SNPs, {result['n_genes_matched']:,} genes matched")

### From a PLINK bimfile (without DonorData)

If you already have a 1000 Genomes reference panel on disk, you can create annotations directly from the `.bim` files — no DonorData object needed.

In [ ]:
from cellink.resources import get_1000genomes_plink_files

plink_path, plink_prefix = get_1000genomes_plink_files(
    config_path="cellink/src/cellink/resources/config/1000genomes.yaml",
    population="EUR",
    return_path=True,
)

for chrom in range(1, 23):
    bimfile = os.path.join(plink_path, f"{plink_prefix}{chrom}.bim")

    result = make_continuous_annot_from_bimfile(
        bimfile=bimfile,
        scores=scores,
        annot_file=f"continuous_annots_1kg/{ct_safe}.{chrom}.annot.gz",
        gene_coord_file=GENE_COORD_FILE,
        windowsize=100_000,
        score_agg="max",
    )
    print(f"chr{chrom}: {result['n_nonzero_snps']:,} non-zero SNPs")

### Binary vs continuous: interchangeable downstream calls

Both `make_annot_from_bimfile` and `make_continuous_annot_from_bimfile` write CHR/BP/SNP/CM/ANNOT format files. The downstream LD-score and heritability commands are identical:

In [ ]:
from cellink.tl.external import make_annot_from_bimfile
from cellink.tl.external import generate_sldsc_genesets

# --- Option A: binary annotation (top 10% genes → 1, rest → 0) ---
summary = generate_sldsc_genesets(specificity, dd.C, out_dir="ldsc_genesets", top_frac=0.10)

for chrom in range(1, 23):
    bimfile = os.path.join(plink_path, f"{plink_prefix}{chrom}.bim")
    make_annot_from_bimfile(
        bimfile=bimfile,
        annot_file=f"binary_annots/{ct_safe}.{chrom}.annot.gz",
        gene_set_file=f"ldsc_genesets/{ct_safe}.GeneSet",
        gene_coord_file=GENE_COORD_FILE,
        windowsize=100_000,
        runner=runner,
    )

# --- Option B: continuous annotation ---
for chrom in range(1, 23):
    bimfile = os.path.join(plink_path, f"{plink_prefix}{chrom}.bim")
    make_continuous_annot_from_bimfile(
        bimfile=bimfile,
        scores=specificity[CELL_TYPE],
        annot_file=f"continuous_annots_1kg/{ct_safe}.{chrom}.annot.gz",
        gene_coord_file=GENE_COORD_FILE,
        windowsize=100_000,
    )

# --- Downstream call is IDENTICAL for both ---
from cellink.resources import get_1000genomes_ld_scores, get_1000genomes_ld_weights
ldscores_path, ldscores_prefix = get_1000genomes_ld_scores(
    config_path="cellink/src/cellink/resources/config/1000genomes.yaml", population="EUR", return_path=True
)
ldweights_path, ldweights_prefix = get_1000genomes_ld_weights(
    config_path="cellink/src/cellink/resources/config/1000genomes.yaml", population="EUR", return_path=True
)

annot_dir = "continuous_annots_1kg"   # or "binary_annots" — same call either way
for chrom in range(1, 23):
    bimfile = os.path.join(plink_path, f"{plink_prefix}{chrom}.bim")
    compute_ld_scores_with_annotations_from_bimfile(
        bfile_prefix=os.path.join(plink_path, f"{plink_prefix}{chrom}"),
        annot_file=f"{annot_dir}/{ct_safe}.{chrom}.annot.gz",
        out_prefix=f"{annot_dir}/{ct_safe}.{chrom}",
        runner=runner,
    )

In [ ]:
# Cell-type-specific heritability — also identical for binary or continuous annots
with open("celltype_ldscores_continuous.txt", "w") as f:
    f.write(f"{ct_safe}\t{annot_dir}/{ct_safe}.\n")

result = estimate_celltype_specific_heritability(
    sumstats_file="fake_munged.sumstats.gz",   # replace with real munged GWAS
    ref_ld_chr=os.path.join(ldscores_path, ldscores_prefix),
    w_ld_chr=os.path.join(ldweights_path, ldweights_prefix),
    ref_ld_chr_cts="celltype_ldscores_continuous.txt",
    out_prefix="h2_cts_continuous",
    run=True,
    runner=runner,
)

---
## Part 2 — MAGMA from Scratch

MAGMA provides a complementary enrichment analysis that does not require LD score files. The pipeline has three steps:

| Step | Function | Output |
|---|---|---|
| I — Annotate | `run_magma_annotate` | `.genes.annot` |
| II — Gene analysis | `run_magma_gene_analysis` | `.genes.raw` |
| III-a — Gene-set analysis | `run_magma_gsa` | `.gsa.out` |
| III-b — Gene property analysis | `run_magma_gpa` | `.gsa.out` |

Steps I and II only need to be run once per GWAS. Step III is run once per score set / method.

### Step I — Annotate SNPs to genes

In [ ]:
# SNP location file: tab-delimited, columns SNP / CHR / BP
# This can be derived from your GWAS summary statistics file.
GWAS_SNP_LOC = "/project/genomics/ayshan/ldsc_analysis/data_2/magma/gwas_snps_scz.txt"

annot_result = run_magma_annotate(
    snp_loc=GWAS_SNP_LOC,
    gene_loc=MAGMA_GENE_LOC,
    out_prefix="magma_results/scz",
    magma_bin=MAGMA_BIN,
    window_kb=0,   # gene body only; use e.g. 35 for ±35 kb
)
print("Annotation file:", annot_result["annot_file"])

In [ ]:
# Preview the command without running:
cmd_preview = run_magma_annotate(
    snp_loc=GWAS_SNP_LOC,
    gene_loc=MAGMA_GENE_LOC,
    out_prefix="magma_results/scz",
    magma_bin=MAGMA_BIN,
    run=False,
)
print(" ".join(cmd_preview["command"]))

### Step II — Gene-level association analysis

This step uses a PLINK LD reference panel to compute gene-level z-scores from the GWAS p-values. The output `.genes.raw` file is the input for both GSA and GPA.

In [ ]:
# GWAS p-value file: tab/space-delimited, requires SNP and P columns
GWAS_PVAL_FILE = "/project/genomics/ayshan/ldsc_analysis/MODEL/INPUT/magma_zscore/scz_gwas.txt"

gene_result = run_magma_gene_analysis(
    bfile=G1000_BFILE,
    pval_file=GWAS_PVAL_FILE,
    gene_annot=annot_result["annot_file"],
    out_prefix="magma_results/scz",
    n_samples=67_390,   # total GWAS sample size
    magma_bin=MAGMA_BIN,
)
print("Gene results:", gene_result["gene_results"])

In [ ]:
# If you already have a pre-computed .genes.raw (e.g. from the combined_pipeline)
# you can skip Steps I and II and point directly to it:
GENES_RAW = "/project/genomics/ayshan/ldsc_analysis/MODEL/INPUT/magma_zscore/scz_result.genes.raw"

---
## Part 3 — MAGMA Gene-Set Analysis (GSA)

GSA tests whether the top-scoring genes in each cell type show stronger GWAS association than background. There are two paths to create the required GMT file:

* **From a scores DataFrame** — use `scores_to_gmt` (selects top-N% per cell type)
* **From existing LDSC `.GeneSet` files** — use `genesets_dir_to_entrez_gmt`

### Path A — From continuous scores

In [ ]:
# specificity is a genes × cell-types DataFrame with ENSG IDs as the index
gmt_file = scores_to_gmt(
    scores=specificity,
    out_file="magma_gsa/onek1k_specificity_top10.gmt",
    top_frac=0.10,          # top 10% genes per cell type
    ascending=False,         # ascending=True would select the bottom 10%
    set_name_prefix="onek1k_specificity",  # optional prefix for set names
)
print("GMT file:", gmt_file)

# Preview the first two lines
with open(gmt_file) as f:
    for i, line in enumerate(f):
        parts = line.rstrip().split("\t")
        print(f"Set '{parts[0]}': {len(parts)-2} genes")
        if i == 1:
            break

In [ ]:
# If gene scores use gene symbols rather than ENSG IDs, provide a mapping:
# gene_map can be a path to a TSV with columns gene_name / ensg_id,
# or a pd.Series indexed by symbol with ENSG values.

# gmt_file = scores_to_gmt(
#     scores=scores_with_symbols,
#     out_file="magma_gsa/scores_mapped.gmt",
#     gene_map="cellink/combined_pipeline/gene_name_to_ensg.tsv",
# )

### Path B — From existing LDSC `.GeneSet` files

In [ ]:
# Converts the *.GeneSet files produced by generate_sldsc_genesets into GMT.
# Gene IDs are passed through as-is (no Ensembl → Entrez conversion).
gmt_from_genesets = genesets_dir_to_entrez_gmt(
    geneset_dir="ldsc_genesets",
    out_gmt="magma_gsa/ldsc_genesets.gmt",
    include_control=False,
)
print("GMT from GeneSet files:", gmt_from_genesets)

### Run GSA

In [ ]:
gsa_result = run_magma_gsa(
    gene_results=GENES_RAW,
    set_annot=str(gmt_file),
    out_prefix="magma_gsa/scz_onek1k_specificity",
    magma_bin=MAGMA_BIN,
)
print("GSA results:", gsa_result["results_file"])

# Load and display results
gsa_df = pd.read_csv(gsa_result["results_file"], sep=r"\s+", comment="#")
gsa_df.sort_values("P").head(10)

In [ ]:
# Extra MAGMA flags are forwarded via **kwargs, e.g. joint competitive test:
# run_magma_gsa(..., model="multi")

---
## Part 4 — MAGMA Gene Property Analysis (GPA)

GPA tests the linear relationship between continuous per-gene scores and GWAS gene z-scores. Unlike GSA it uses all genes with scores — no top-N threshold. The covariate file contains one column per cell type.

### Build the covariate file

In [ ]:
covar_file = scores_to_covar(
    scores=specificity,
    out_file="magma_gpa/onek1k_specificity.covar",
    negate=False,   # set negate=True to test enrichment in low-score genes
)
print("Covariate file:", covar_file)

# Preview
pd.read_csv(covar_file, sep="\t", index_col=0).head()

In [ ]:
# Gene symbols → ENSG mapping works the same way as in scores_to_gmt:
# covar_file = scores_to_covar(
#     scores=scores_with_symbols,
#     out_file="magma_gpa/scores_mapped.covar",
#     gene_map="cellink/combined_pipeline/gene_name_to_ensg.tsv",
# )

### Joint GPA (default)

All cell types are tested in a single MAGMA call. MAGMA may drop highly correlated covariates via its collinearity filter. Use univariate mode if this is a concern.

In [ ]:
gpa_result = run_magma_gpa(
    gene_results=GENES_RAW,
    gene_covar=str(covar_file),
    out_prefix="magma_gpa/scz_onek1k_specificity_joint",
    magma_bin=MAGMA_BIN,
    univariate=False,
)
print("GPA results:", gpa_result["results_file"])

gpa_df = pd.read_csv(gpa_result["results_file"], sep=r"\s+", comment="#")
gpa_df.sort_values("P").head(10)

### Univariate GPA

Each cell type is tested in an independent MAGMA call. Slower than joint mode but guarantees a result for every cell type — recommended when covariate columns are highly correlated (e.g. residual CV scores or negated scores).

In [ ]:
gpa_univ_result = run_magma_gpa(
    gene_results=GENES_RAW,
    gene_covar=str(covar_file),
    out_prefix="magma_gpa/scz_onek1k_specificity_univariate",
    magma_bin=MAGMA_BIN,
    univariate=True,
)
print("Univariate GPA results:", gpa_univ_result["results_file"])

gpa_univ_df = pd.read_csv(gpa_univ_result["results_file"], sep=r"\s+", comment="#")
gpa_univ_df.sort_values("P").head(10)

---
## Part 5 — LDSC → MAGMA: Full Workflows

The two complete workflows from LDSC score outputs to MAGMA results.

### Workflow A: LDSC binary genesets → MAGMA GSA

In [ ]:
# 1. Generate LDSC gene sets (top 10% specificity per cell type)
generate_sldsc_genesets(specificity, dd.C, out_dir="ldsc_genesets", top_frac=0.10)

# 2. Convert .GeneSet → GMT (MAGMA format, ENSG IDs kept as-is)
gmt_ldsc = genesets_dir_to_entrez_gmt(
    geneset_dir="ldsc_genesets",
    out_gmt="magma_gsa/ldsc_genesets_workflow_a.gmt",
)

# 3. GSA
gsa_a = run_magma_gsa(
    gene_results=GENES_RAW,
    set_annot=str(gmt_ldsc),
    out_prefix="magma_gsa/workflow_a_scz",
    magma_bin=MAGMA_BIN,
)

results_a = pd.read_csv(gsa_a["results_file"], sep=r"\s+", comment="#")
print(results_a.sort_values("P").head())

### Workflow B: Continuous scores → MAGMA GPA

In [ ]:
# 1. Build covariate file directly from the specificity score matrix
covar_b = scores_to_covar(
    scores=specificity,
    out_file="magma_gpa/onek1k_specificity_workflow_b.covar",
)

# 2. GPA (joint mode)
gpa_b = run_magma_gpa(
    gene_results=GENES_RAW,
    gene_covar=str(covar_b),
    out_prefix="magma_gpa/workflow_b_scz",
    magma_bin=MAGMA_BIN,
)

results_b = pd.read_csv(gpa_b["results_file"], sep=r"\s+", comment="#")
print(results_b.sort_values("P").head())

---
## Part 6 — Tips and Common Patterns

### Previewing commands without running

All MAGMA functions accept `run=False` to return the command list without executing it. Useful for debugging or submitting to a cluster.

In [ ]:
for fn_name, result in [
    ("run_magma_annotate",
     run_magma_annotate(snp_loc=GWAS_SNP_LOC, gene_loc=MAGMA_GENE_LOC,
                        out_prefix="magma_results/scz", magma_bin=MAGMA_BIN, run=False)),
    ("run_magma_gene_analysis",
     run_magma_gene_analysis(bfile=G1000_BFILE, pval_file=GWAS_PVAL_FILE,
                             gene_annot="magma_results/scz.genes.annot",
                             out_prefix="magma_results/scz",
                             n_samples=67_390, magma_bin=MAGMA_BIN, run=False)),
    ("run_magma_gsa",
     run_magma_gsa(gene_results=GENES_RAW, set_annot=str(gmt_file),
                   out_prefix="magma_gsa/preview", magma_bin=MAGMA_BIN, run=False)),
    ("run_magma_gpa",
     run_magma_gpa(gene_results=GENES_RAW, gene_covar=str(covar_file),
                   out_prefix="magma_gpa/preview", magma_bin=MAGMA_BIN, run=False)),
]:
    print(f"{fn_name}:\n  {' '.join(result['command'])}\n")

### Negating scores to test low-score enrichment

Some metrics (e.g. coefficient of variation) are expected to be *low* in disease-relevant genes. Pass `negate=True` to `scores_to_covar`, or `ascending=True` to `scores_to_gmt`, to select/test genes with the *lowest* scores:

In [ ]:
# GSA: bottom 10% genes (lowest CV)
gmt_bottom = scores_to_gmt(
    scores=specificity,
    out_file="magma_gsa/specificity_bottom10.gmt",
    top_frac=0.10,
    ascending=True,   # select lowest-scoring genes
)

# GPA: negate scores so low-score genes get high covariate values
covar_neg = scores_to_covar(
    scores=specificity,
    out_file="magma_gpa/specificity_negated.covar",
    negate=True,
)

### Passing extra MAGMA flags

Any MAGMA option not explicitly supported can be forwarded via `**kwargs`. Each `key=value` pair becomes `--key value` in the MAGMA call:

In [ ]:
# Competitive gene-set test (MAGMA's "multi" model)
run_magma_gsa(
    gene_results=GENES_RAW,
    set_annot=str(gmt_file),
    out_prefix="magma_gsa/scz_competitive",
    magma_bin=MAGMA_BIN,
    model="multi",   # forwarded as --model multi
    run=False,       # preview only
)

---
## Summary

This tutorial demonstrated all new functions added to `cellink.tl.external`:

| Function | Purpose |
|---|---|
| `make_continuous_annot_from_bimfile` | Continuous S-LDSC annotation from a PLINK `.bim` file |
| `make_continuous_annot_from_donor_data` | Continuous S-LDSC annotation from a `DonorData` object |
| `scores_to_gmt` | Convert score DataFrame → MAGMA GMT (top-N% per cell type) |
| `scores_to_covar` | Convert score DataFrame → MAGMA `.covar` (all genes, continuous) |
| `genesets_dir_to_entrez_gmt` | Convert LDSC `.GeneSet` files → MAGMA GMT |
| `run_magma_annotate` | MAGMA Step I: SNP → gene annotation |
| `run_magma_gene_analysis` | MAGMA Step II: gene-level association analysis |
| `run_magma_gsa` | MAGMA Step III-a: gene-set analysis (binary) |
| `run_magma_gpa` | MAGMA Step III-b: gene property analysis (continuous), with optional univariate mode |

Key properties shared across all functions:

* **`run=False`** returns the command list without executing — useful for HPC job submission.
* **Gene ID handling** — `scores_to_gmt` and `scores_to_covar` accept a `gene_map` argument for symbol → ENSG translation; rows that cannot be mapped to ENSG are dropped.
* **Format compatibility** — `make_continuous_annot_from_*` and `make_annot_from_*` write the same five-column file so all downstream LDSC calls are identical.
* **Univariate GPA** — `run_magma_gpa(univariate=True)` tests each cell type independently, avoiding MAGMA's collinearity filter when scores are highly correlated.